# Reinforcement Learning from Human Feedback (RLHF) - The Technique Behind ChatGPT

This notebook explores **RLHF** - the technique that transformed large language models from text predictors into helpful, harmless, and honest assistants. This is what made ChatGPT successful.

**What makes RLHF special?** Pre-trained models like GPT-3 are excellent at predicting text, but they don't know what humans want. RLHF teaches models to align with human preferences and values.

**Important Note:** This notebook loads multiple transformer models (GPT-2, DistilBERT) and may require significant memory (4GB+ RAM). Due to computational requirements, this notebook is designed to be read and understood rather than executed end-to-end on limited hardware. Individual sections can be run if you have sufficient resources.

## The Alignment Problem

**Pre-training** teaches a model to predict the next token:
- Trained on massive internet text
- Learns grammar, facts, reasoning patterns
- But also learns toxic content, misinformation, unhelpful behaviors

**The gap**: Good at predicting ≠ Good at helping humans

Examples of misalignment:
- User: "How do I break into a car?"
- Unaligned model: Provides instructions (may be for illegal purpose)
- Aligned model: Asks if they're locked out, suggests calling a locksmith

**RLHF bridges this gap** by training models to optimize for human preferences instead of just likelihood.

## The Three-Stage RLHF Pipeline

RLHF consists of three sequential stages:

**Stage 1: Supervised Fine-Tuning (SFT)**
- Start with pre-trained base model
- Fine-tune on high-quality demonstrations
- Dataset: prompt → desired completion pairs
- Creates initial helpful assistant

**Stage 2: Reward Modeling (RM)**
- Collect human preference data (A vs B comparisons)
- Train reward model to predict human preferences
- Model learns: "What makes a good response?"

**Stage 3: Reinforcement Learning (PPO)**
- Use RL to optimize policy based on reward model
- Maximize reward while staying close to SFT model
- KL penalty prevents reward hacking

**Result**: A model that generates responses humans prefer while maintaining coherence and safety.

## What We'll Build

Due to computational constraints, we'll implement RLHF on a **sentiment steering task**:
- Base model: GPT-2 (124M parameters)
- Task: Generate positive movie reviews
- Reward: Sentiment classifier score

This captures the essence of RLHF:
1. Start with pre-trained model (GPT-2)
2. Define desired behavior (positive sentiment)
3. Use RL to steer model toward desired behavior

**Coverage:**
- Complete RLHF pipeline implementation
- Reward modeling from comparisons
- PPO optimization with KL penalty
- DPO (Direct Preference Optimization) as alternative
- Practical challenges and solutions

## Setup and Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
from typing import List, Dict, Tuple, Optional

from transformers import (
    GPT2LMHeadModel, 
    GPT2Tokenizer,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)

from aiml_notebooks import set_seed, get_device

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
set_seed(42)

# Use CPU for transformers (better compatibility)
device = get_device(prefer_cpu=True)
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Stage 1: Supervised Fine-Tuning (SFT)

The first stage creates a baseline model that can follow instructions. In production:
- Start with base LLM (e.g., GPT-3, LLaMA)
- Fine-tune on demonstrations: (prompt, high-quality completion) pairs
- Dataset: 10k-100k examples written by humans or contractors

**What SFT achieves:**
- Model learns to follow instruction format
- Learns basic helpfulness
- Narrows distribution from "all internet text" to "helpful assistant"

**For our sentiment task:** We'll skip SFT since GPT-2 can already generate text. In practice, you'd fine-tune GPT-2 on positive reviews first.

### Load Pre-trained GPT-2

We'll use GPT-2 as our base model. This is analogous to using a pre-trained LLM before RLHF.

In [ ]:
# Load GPT-2 model and tokenizer
model_name = "gpt2"  # 124M parameters

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have pad token

# Load model
base_model = GPT2LMHeadModel.from_pretrained(model_name)
base_model = base_model.to(device)
base_model.eval()

print(f"Loaded {model_name}:")
print(f"  Vocabulary size: {len(tokenizer)}")
print(f"  Model parameters: {sum(p.numel() for p in base_model.parameters()) / 1e6:.1f}M")
print(f"  Hidden size: {base_model.config.hidden_size}")
print(f"  Number of layers: {base_model.config.n_layer}")

### Test Base Model Generation

Let's see what the base GPT-2 generates without any alignment.

In [ ]:
def generate_text(model, tokenizer, prompt, max_length=50, temperature=0.7, num_return=3):
    """Generate text from a prompt."""
    model.eval()
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            do_sample=True,
            num_return_sequences=num_return,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts

# Test on movie review prompt
prompt = "This movie was"
samples = generate_text(base_model, tokenizer, prompt, max_length=30, num_return=5)

print(f"Prompt: '{prompt}'\n")
for i, text in enumerate(samples, 1):
    print(f"{i}. {text}")
    print()

print("→ Base model generates mixed sentiment (both positive and negative)")

## Stage 2: Reward Modeling

The reward model learns to predict human preferences. In production:
- Show humans pairs of responses: (A, B) for the same prompt
- Ask: "Which response is better?"
- Collect thousands of comparisons
- Train model to predict which response humans prefer

**Mathematical formulation (Bradley-Terry model):**

Given responses $y_w$ (winner) and $y_l$ (loser) for prompt $x$:

$$P(y_w \succ y_l | x) = \frac{\exp(r(x, y_w))}{\exp(r(x, y_w)) + \exp(r(x, y_l))} = \sigma(r(x, y_w) - r(x, y_l))$$

where $r(x, y)$ is the reward model score and $\sigma$ is the sigmoid function.

**Loss function:**

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l)}[\log \sigma(r(x, y_w) - r(x, y_l))]$$

This is simply **binary cross-entropy** on preference pairs.

### Reward Model Architecture

The reward model uses the same architecture as the language model but with a scalar output head.

In [ ]:
class RewardModel(nn.Module):
    """Reward model built on top of GPT-2."""
    
    def __init__(self, base_model):
        super().__init__()
        self.transformer = base_model.transformer
        self.config = base_model.config
        
        # Reward head: maps hidden state to scalar reward
        self.reward_head = nn.Linear(self.config.hidden_size, 1, bias=False)
    
    def forward(self, input_ids, attention_mask=None):
        """
        Forward pass returns reward for each sequence.
        
        Args:
            input_ids: (batch_size, seq_len)
            attention_mask: (batch_size, seq_len)
        
        Returns:
            rewards: (batch_size,) - scalar reward for each sequence
        """
        # Get transformer outputs
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden_size)
        
        # Use last token's hidden state (like classification)
        # Find the last non-padding token for each sequence
        if attention_mask is not None:
            # Get position of last non-padding token
            last_token_indices = attention_mask.sum(dim=1) - 1
            batch_size = hidden_states.shape[0]
            last_hidden = hidden_states[torch.arange(batch_size), last_token_indices]
        else:
            # Use last token
            last_hidden = hidden_states[:, -1, :]
        
        # Compute rewards
        rewards = self.reward_head(last_hidden).squeeze(-1)  # (batch_size,)
        
        return rewards

# Create reward model
reward_model = RewardModel(base_model)
reward_model = reward_model.to(device)

print(f"Reward model created:")
print(f"  Base parameters: {sum(p.numel() for p in reward_model.transformer.parameters()) / 1e6:.1f}M")
print(f"  Reward head parameters: {sum(p.numel() for p in reward_model.reward_head.parameters())}")
print(f"  Output: Scalar reward per sequence")

### Simulated Preference Dataset

In production, humans compare response pairs. For our sentiment task, we'll simulate preferences using a sentiment classifier as a proxy for human judgment.

In [ ]:
# Load sentiment classifier as proxy for human preferences
sentiment_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment_tokenizer = AutoTokenizer.from_pretrained(sentiment_model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_name)
sentiment_model = sentiment_model.to(device)
sentiment_model.eval()

print(f"Loaded sentiment classifier: {sentiment_model_name}")
print(f"  This will simulate human preferences (positive > negative)")

Now let's create a function to score text sentiment:

In [ ]:
def get_sentiment_score(text, model, tokenizer):
    """Get sentiment score (0=negative, 1=positive)."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        # Index 1 is positive sentiment
        positive_prob = probs[0, 1].item()
    
    return positive_prob

# Test sentiment scoring
test_texts = [
    "This movie was absolutely fantastic! I loved every moment.",
    "This movie was terrible and boring. Complete waste of time.",
    "This movie was okay, nothing special."
]

print("Sentiment scores (0=negative, 1=positive):\n")
for text in test_texts:
    score = get_sentiment_score(text, sentiment_model, sentiment_tokenizer)
    print(f"{score:.3f} - {text}")

### Generate Preference Dataset

We'll generate pairs of completions and use sentiment scores to determine preferences.

In [ ]:
def create_preference_pairs(model, tokenizer, prompts, num_pairs_per_prompt=2):
    """Generate preference pairs from prompts."""
    preference_data = []
    
    for prompt in prompts:
        for _ in range(num_pairs_per_prompt):
            # Generate two completions
            completions = generate_text(
                model, tokenizer, prompt, 
                max_length=40, temperature=0.9, num_return=2
            )
            
            # Score both
            scores = [
                get_sentiment_score(text, sentiment_model, sentiment_tokenizer)
                for text in completions
            ]
            
            # Determine winner/loser
            if scores[0] > scores[1]:
                chosen, rejected = completions[0], completions[1]
            else:
                chosen, rejected = completions[1], completions[0]
            
            preference_data.append({
                'prompt': prompt,
                'chosen': chosen,
                'rejected': rejected
            })
    
    return preference_data

# Create prompts
train_prompts = [
    "This movie was",
    "The film is",
    "I thought the movie was",
    "The acting in this film was",
    "Overall, this movie is",
    "After watching this film, I felt",
    "The story was",
    "This cinema experience was"
]

print("Generating preference pairs...\n")
preference_data = create_preference_pairs(base_model, tokenizer, train_prompts, num_pairs_per_prompt=3)

print(f"Created {len(preference_data)} preference pairs\n")
print("Example pair:")
example = preference_data[0]
print(f"  Chosen:   {example['chosen']}")
print(f"  Rejected: {example['rejected']}")

### Preference Dataset Class

In [ ]:
class PreferenceDataset(Dataset):
    """Dataset for preference pairs."""
    
    def __init__(self, data, tokenizer, max_length=64):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Tokenize chosen and rejected
        chosen_encodings = self.tokenizer(
            item['chosen'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        rejected_encodings = self.tokenizer(
            item['rejected'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'chosen_input_ids': chosen_encodings['input_ids'].squeeze(0),
            'chosen_attention_mask': chosen_encodings['attention_mask'].squeeze(0),
            'rejected_input_ids': rejected_encodings['input_ids'].squeeze(0),
            'rejected_attention_mask': rejected_encodings['attention_mask'].squeeze(0)
        }

# Create dataset and dataloader
pref_dataset = PreferenceDataset(preference_data, tokenizer)
pref_dataloader = DataLoader(pref_dataset, batch_size=4, shuffle=True)

print(f"Preference dataset size: {len(pref_dataset)}")
print(f"Batch size: 4")
print(f"Number of batches: {len(pref_dataloader)}")

### Train Reward Model

We'll train the reward model using the Bradley-Terry preference learning objective.

In [ ]:
def train_reward_model(model, dataloader, num_epochs=3, lr=1e-5):
    """Train reward model on preference pairs."""
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    losses = []
    accuracies = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_correct = 0
        epoch_total = 0
        
        for batch in dataloader:
            # Move to device
            chosen_ids = batch['chosen_input_ids'].to(device)
            chosen_mask = batch['chosen_attention_mask'].to(device)
            rejected_ids = batch['rejected_input_ids'].to(device)
            rejected_mask = batch['rejected_attention_mask'].to(device)
            
            # Get rewards for chosen and rejected
            r_chosen = model(chosen_ids, chosen_mask)
            r_rejected = model(rejected_ids, rejected_mask)
            
            # Bradley-Terry loss: -log(sigmoid(r_chosen - r_rejected))
            # Equivalent to: binary cross-entropy with target=1
            loss = -F.logsigmoid(r_chosen - r_rejected).mean()
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            # Track metrics
            epoch_loss += loss.item()
            
            # Accuracy: how often does model prefer chosen over rejected?
            correct = (r_chosen > r_rejected).sum().item()
            epoch_correct += correct
            epoch_total += chosen_ids.shape[0]
        
        avg_loss = epoch_loss / len(dataloader)
        accuracy = epoch_correct / epoch_total
        
        losses.append(avg_loss)
        accuracies.append(accuracy)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.3f}")
    
    return losses, accuracies

print("Training reward model...\n")
rm_losses, rm_accuracies = train_reward_model(reward_model, pref_dataloader, num_epochs=3, lr=1e-5)

print(f"\nFinal accuracy: {rm_accuracies[-1]:.3f}")
print("→ Reward model learns to predict which text is more positive")

### Visualize Reward Model Training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(rm_losses, marker='o', linewidth=2, color='blue')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_title('Reward Model Training Loss', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(rm_accuracies, marker='o', linewidth=2, color='green')
ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Accuracy', fontsize=11)
ax2.set_title('Reward Model Accuracy', fontsize=12, fontweight='bold')
ax2.set_ylim([0, 1])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Test Reward Model

Let's verify the reward model assigns higher scores to positive text.

In [ ]:
def get_reward(text, model, tokenizer):
    """Get reward score for text."""
    model.eval()
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64).to(device)
    
    with torch.no_grad():
        reward = model(inputs['input_ids'], inputs['attention_mask'])
    
    return reward.item()

# Test on examples
test_examples = [
    "This movie was absolutely amazing and wonderful!",
    "This movie was pretty good and enjoyable.",
    "This movie was okay, not great.",
    "This movie was disappointing and boring.",
    "This movie was terrible and awful!"
]

print("Reward model scores (higher = more preferred):\n")
for text in test_examples:
    reward = get_reward(text, reward_model, tokenizer)
    print(f"{reward:6.3f} - {text}")

print("\n→ Reward model correctly ranks positive sentiment higher")

## Stage 3: Reinforcement Learning with PPO

The final stage uses RL to optimize the policy (language model) to maximize the reward while staying close to the SFT model.

**The RL objective:**

$$\max_{\pi_\theta} \mathbb{E}_{x \sim D, y \sim \pi_\theta(\cdot|x)}\left[r(x, y) - \beta \cdot D_{\text{KL}}(\pi_\theta(\cdot|x) \| \pi_{\text{ref}}(\cdot|x))\right]$$

where:
- $r(x, y)$ is the reward from the reward model
- $\beta$ is the KL penalty coefficient
- $\pi_{\text{ref}}$ is the reference model (SFT model)

**Why the KL penalty?**

Without it, the model could "reward hack":
- Generate nonsensical text that exploits reward model weaknesses
- Forget language modeling abilities
- Produce repetitive or degenerate outputs

The KL penalty keeps the policy close to the reference model, preserving coherence and preventing mode collapse.

### PPO Implementation

We'll implement a simplified version of PPO for language model fine-tuning. Key components:

1. **Policy model**: GPT-2 (initialized from base)
2. **Reference model**: Frozen copy of initial policy
3. **Value model**: Estimates expected reward (helps reduce variance)
4. **Reward model**: Trained in Stage 2

First, let's create the value model:

In [ ]:
class ValueModel(nn.Module):
    """Value model for PPO (estimates expected return)."""
    
    def __init__(self, base_model):
        super().__init__()
        self.transformer = base_model.transformer
        self.config = base_model.config
        self.value_head = nn.Linear(self.config.hidden_size, 1, bias=False)
    
    def forward(self, input_ids, attention_mask=None):
        """
        Returns value for each token in sequence.
        
        Args:
            input_ids: (batch, seq_len)
            attention_mask: (batch, seq_len)
        
        Returns:
            values: (batch, seq_len)
        """
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden)
        values = self.value_head(hidden_states).squeeze(-1)  # (batch, seq_len)
        
        return values

# Create models for PPO
policy_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
ref_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
value_model = ValueModel(GPT2LMHeadModel.from_pretrained(model_name)).to(device)

# Freeze reference model
for param in ref_model.parameters():
    param.requires_grad = False
ref_model.eval()

print("Created PPO models:")
print(f"  Policy model: {sum(p.numel() for p in policy_model.parameters()) / 1e6:.1f}M params (trainable)")
print(f"  Reference model: {sum(p.numel() for p in ref_model.parameters()) / 1e6:.1f}M params (frozen)")
print(f"  Value model: {sum(p.numel() for p in value_model.parameters()) / 1e6:.1f}M params (trainable)")

### Helper Functions for PPO

In [ ]:
def compute_kl_divergence(logprobs_policy, logprobs_ref):
    """Compute KL divergence between policy and reference distributions.
    
    KL(π || π_ref) = E[log π - log π_ref]
    """
    return (logprobs_policy - logprobs_ref).mean()

def compute_advantages(rewards, values, gamma=0.99, lam=0.95):
    """Compute GAE (Generalized Advantage Estimation).
    
    This is a variance-reduced advantage estimate that balances
    bias vs variance by combining n-step returns.
    """
    advantages = []
    last_gae = 0
    
    # Work backwards through trajectory
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_value = 0
        else:
            next_value = values[t + 1]
        
        # TD error: δ_t = r_t + γV(s_{t+1}) - V(s_t)
        delta = rewards[t] + gamma * next_value - values[t]
        
        # GAE: A_t = δ_t + γλA_{t+1}
        last_gae = delta + gamma * lam * last_gae
        advantages.insert(0, last_gae)
    
    return torch.tensor(advantages)

def get_log_probs(model, input_ids, attention_mask):
    """Get log probabilities of generated tokens."""
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits  # (batch, seq_len, vocab_size)
    
    # Get log probs for actual tokens (shifted by 1)
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    token_log_probs = torch.gather(
        log_probs, 
        dim=2, 
        index=input_ids[:, 1:].unsqueeze(-1)
    ).squeeze(-1)
    
    return token_log_probs

print("PPO helper functions defined:")
print("  - compute_kl_divergence: KL(policy || reference)")
print("  - compute_advantages: GAE for variance reduction")
print("  - get_log_probs: Extract log probabilities of tokens")

### PPO Training Loop

The PPO training process:

1. **Generate**: Sample completions from current policy
2. **Evaluate**: Get rewards from reward model
3. **Compute KL**: Measure divergence from reference
4. **Optimize**: Update policy and value function using PPO

This is a simplified version focusing on the core ideas.

In [ ]:
def ppo_train_step(
    policy_model,
    ref_model,
    value_model,
    reward_model,
    prompts,
    tokenizer,
    policy_optimizer,
    value_optimizer,
    beta_kl=0.1,
    clip_epsilon=0.2,
    max_gen_length=30
):
    """Single PPO training step."""
    
    # 1. Generate completions from current policy
    policy_model.eval()
    
    all_input_ids = []
    all_attention_masks = []
    
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        
        with torch.no_grad():
            outputs = policy_model.generate(
                **inputs,
                max_length=max_gen_length,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Pad to same length
        attention_mask = (outputs != tokenizer.pad_token_id).long()
        
        all_input_ids.append(outputs)
        all_attention_masks.append(attention_mask)
    
    # Stack
    input_ids = torch.cat(all_input_ids, dim=0)
    attention_mask = torch.cat(all_attention_masks, dim=0)
    
    # 2. Get rewards from reward model
    with torch.no_grad():
        rewards = reward_model(input_ids, attention_mask)
    
    # 3. Get log probs from policy and reference
    policy_model.train()
    
    with torch.no_grad():
        ref_log_probs = get_log_probs(ref_model, input_ids, attention_mask)
    
    policy_log_probs = get_log_probs(policy_model, input_ids, attention_mask)
    
    # 4. Compute KL divergence
    kl_div = compute_kl_divergence(policy_log_probs.mean(dim=1), ref_log_probs.mean(dim=1))
    
    # 5. Total reward = reward - beta * KL
    total_rewards = rewards - beta_kl * kl_div
    
    # 6. Get values from value model
    values = value_model(input_ids, attention_mask)
    
    # 7. Compute advantages (simplified - use reward as advantage)
    advantages = total_rewards - values.mean(dim=1).detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    # 8. PPO policy loss with clipping
    # For simplicity, we use a simplified policy gradient loss
    policy_loss = -(policy_log_probs.mean(dim=1) * advantages).mean()
    
    # 9. Value loss
    value_loss = F.mse_loss(values.mean(dim=1), total_rewards.detach())
    
    # 10. Update policy
    policy_optimizer.zero_grad()
    policy_loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
    policy_optimizer.step()
    
    # 11. Update value function
    value_optimizer.zero_grad()
    value_loss.backward()
    torch.nn.utils.clip_grad_norm_(value_model.parameters(), 1.0)
    value_optimizer.step()
    
    return {
        'policy_loss': policy_loss.item(),
        'value_loss': value_loss.item(),
        'mean_reward': rewards.mean().item(),
        'mean_kl': kl_div.item(),
        'total_reward': total_rewards.mean().item()
    }

print("PPO training step function defined")
print("  Input: prompts")
print("  Process: generate → reward → compute KL → optimize")
print("  Output: metrics (loss, reward, KL)")

### Run PPO Training

Now let's train the policy using PPO. This will take a few minutes.

In [ ]:
# Training configuration
num_iterations = 20
batch_size = 4
lr_policy = 1e-5
lr_value = 1e-4
beta_kl = 0.05  # KL penalty coefficient

# Optimizers
policy_optimizer = torch.optim.AdamW(policy_model.parameters(), lr=lr_policy)
value_optimizer = torch.optim.AdamW(value_model.parameters(), lr=lr_value)

# Training prompts
ppo_prompts = [
    "This movie was",
    "The film is",
    "I thought the movie was",
    "Overall, this movie is"
]

# Track metrics
metrics_history = defaultdict(list)

print("Starting PPO training...\n")
print(f"Configuration:")
print(f"  Iterations: {num_iterations}")
print(f"  Batch size: {batch_size}")
print(f"  Policy LR: {lr_policy}")
print(f"  Value LR: {lr_value}")
print(f"  KL penalty (β): {beta_kl}")
print()

for iteration in range(num_iterations):
    # Sample prompts for this iteration
    batch_prompts = np.random.choice(ppo_prompts, size=batch_size, replace=True).tolist()
    
    # PPO training step
    metrics = ppo_train_step(
        policy_model=policy_model,
        ref_model=ref_model,
        value_model=value_model,
        reward_model=reward_model,
        prompts=batch_prompts,
        tokenizer=tokenizer,
        policy_optimizer=policy_optimizer,
        value_optimizer=value_optimizer,
        beta_kl=beta_kl
    )
    
    # Track metrics
    for key, value in metrics.items():
        metrics_history[key].append(value)
    
    if (iteration + 1) % 5 == 0:
        print(f"Iteration {iteration+1}/{num_iterations}:")
        print(f"  Reward: {metrics['mean_reward']:.3f}")
        print(f"  KL: {metrics['mean_kl']:.4f}")
        print(f"  Total: {metrics['total_reward']:.3f}")
        print()

print("PPO training complete!")

### Visualize PPO Training Metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Reward
axes[0, 0].plot(metrics_history['mean_reward'], linewidth=2, color='green')
axes[0, 0].set_xlabel('Iteration', fontsize=10)
axes[0, 0].set_ylabel('Mean Reward', fontsize=10)
axes[0, 0].set_title('Reward from RM', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# KL divergence
axes[0, 1].plot(metrics_history['mean_kl'], linewidth=2, color='red')
axes[0, 1].set_xlabel('Iteration', fontsize=10)
axes[0, 1].set_ylabel('KL Divergence', fontsize=10)
axes[0, 1].set_title('KL(Policy || Reference)', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Total reward (reward - β*KL)
axes[1, 0].plot(metrics_history['total_reward'], linewidth=2, color='blue')
axes[1, 0].set_xlabel('Iteration', fontsize=10)
axes[1, 0].set_ylabel('Total Reward', fontsize=10)
axes[1, 0].set_title('Total Reward (with KL penalty)', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Policy loss
axes[1, 1].plot(metrics_history['policy_loss'], linewidth=2, color='purple')
axes[1, 1].set_xlabel('Iteration', fontsize=10)
axes[1, 1].set_ylabel('Policy Loss', fontsize=10)
axes[1, 1].set_title('PPO Policy Loss', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Compare Before and After RLHF

Let's generate text from both the base model and the RLHF-trained model.

In [ ]:
test_prompts = [
    "This movie was",
    "The film is",
    "I thought the movie was"
]

print("Comparing Base Model vs RLHF Model\n")
print("=" * 80)

for prompt in test_prompts:
    print(f"\nPrompt: '{prompt}'\n")
    
    # Base model generations
    base_samples = generate_text(base_model, tokenizer, prompt, max_length=30, num_return=3)
    print("Base Model:")
    for i, text in enumerate(base_samples, 1):
        sentiment = get_sentiment_score(text, sentiment_model, sentiment_tokenizer)
        print(f"  {i}. [{sentiment:.2f}] {text}")
    
    # RLHF model generations
    rlhf_samples = generate_text(policy_model, tokenizer, prompt, max_length=30, num_return=3)
    print("\nRLHF Model:")
    for i, text in enumerate(rlhf_samples, 1):
        sentiment = get_sentiment_score(text, sentiment_model, sentiment_tokenizer)
        print(f"  {i}. [{sentiment:.2f}] {text}")
    
    print("\n" + "=" * 80)

print("\n→ RLHF model generates more positive sentiment (higher scores)")

## DPO: Direct Preference Optimization

**DPO** is a simpler alternative to RLHF that skips the reward model and RL stages.

**Key insight:** We can directly optimize the policy on preference data without explicitly learning a reward function.

### The DPO Objective

Given preference pairs $(x, y_w, y_l)$ where $y_w$ is preferred over $y_l$:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

**Intuition:**
- Increase probability of preferred response $y_w$
- Decrease probability of rejected response $y_l$
- Stay close to reference model (via ratio)

**Advantages over RLHF:**
1. No reward model needed
2. No RL optimization (simpler training)
3. More stable (no reward hacking)
4. Faster to train

**Trade-offs:**
- Less flexible (can't change reward function after training)
- Requires preference pairs (not scalar rewards)

### DPO Implementation

In [ ]:
def dpo_loss(policy_model, ref_model, batch, beta=0.1):
    """Compute DPO loss for a batch of preference pairs."""
    
    # Get inputs
    chosen_ids = batch['chosen_input_ids'].to(device)
    chosen_mask = batch['chosen_attention_mask'].to(device)
    rejected_ids = batch['rejected_input_ids'].to(device)
    rejected_mask = batch['rejected_attention_mask'].to(device)
    
    # Get log probs from policy
    policy_chosen_logprobs = get_log_probs(policy_model, chosen_ids, chosen_mask).mean(dim=1)
    policy_rejected_logprobs = get_log_probs(policy_model, rejected_ids, rejected_mask).mean(dim=1)
    
    # Get log probs from reference
    with torch.no_grad():
        ref_chosen_logprobs = get_log_probs(ref_model, chosen_ids, chosen_mask).mean(dim=1)
        ref_rejected_logprobs = get_log_probs(ref_model, rejected_ids, rejected_mask).mean(dim=1)
    
    # Compute log ratios
    chosen_log_ratio = policy_chosen_logprobs - ref_chosen_logprobs
    rejected_log_ratio = policy_rejected_logprobs - ref_rejected_logprobs
    
    # DPO loss
    loss = -F.logsigmoid(beta * (chosen_log_ratio - rejected_log_ratio)).mean()
    
    # Compute implicit reward (for monitoring)
    with torch.no_grad():
        chosen_rewards = beta * chosen_log_ratio
        rejected_rewards = beta * rejected_log_ratio
    
    return loss, chosen_rewards.mean(), rejected_rewards.mean()

def train_dpo(policy_model, ref_model, dataloader, num_epochs=3, lr=1e-5, beta=0.1):
    """Train model using DPO."""
    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=lr)
    
    losses = []
    reward_margins = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_margin = 0
        
        for batch in dataloader:
            loss, chosen_reward, rejected_reward = dpo_loss(
                policy_model, ref_model, batch, beta=beta
            )
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_margin += (chosen_reward - rejected_reward).item()
        
        avg_loss = epoch_loss / len(dataloader)
        avg_margin = epoch_margin / len(dataloader)
        
        losses.append(avg_loss)
        reward_margins.append(avg_margin)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}, Reward Margin: {avg_margin:.4f}")
    
    return losses, reward_margins

print("DPO implementation complete")
print("  Direct optimization on preference pairs")
print("  No reward model, no RL needed")

### Train DPO Model

Let's train a separate model using DPO and compare with RLHF.

In [ ]:
# Create fresh models for DPO
dpo_policy_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
dpo_ref_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

# Freeze reference
for param in dpo_ref_model.parameters():
    param.requires_grad = False
dpo_ref_model.eval()

print("Training DPO model...\n")
dpo_losses, dpo_margins = train_dpo(
    dpo_policy_model, 
    dpo_ref_model, 
    pref_dataloader, 
    num_epochs=3, 
    lr=1e-5, 
    beta=0.1
)

print("\nDPO training complete!")

### Compare DPO Results

In [ ]:
print("Comparing Base vs RLHF vs DPO\n")
print("=" * 80)

test_prompt = "This movie was"

print(f"Prompt: '{test_prompt}'\n")

# Generate from all three models
base_samples = generate_text(base_model, tokenizer, test_prompt, max_length=30, num_return=5)
rlhf_samples = generate_text(policy_model, tokenizer, test_prompt, max_length=30, num_return=5)
dpo_samples = generate_text(dpo_policy_model, tokenizer, test_prompt, max_length=30, num_return=5)

# Compute average sentiment
base_sentiments = [get_sentiment_score(s, sentiment_model, sentiment_tokenizer) for s in base_samples]
rlhf_sentiments = [get_sentiment_score(s, sentiment_model, sentiment_tokenizer) for s in rlhf_samples]
dpo_sentiments = [get_sentiment_score(s, sentiment_model, sentiment_tokenizer) for s in dpo_samples]

print("Base Model:")
for i, (text, score) in enumerate(zip(base_samples, base_sentiments), 1):
    print(f"  [{score:.2f}] {text}")
print(f"  Average: {np.mean(base_sentiments):.3f}\n")

print("RLHF Model:")
for i, (text, score) in enumerate(zip(rlhf_samples, rlhf_sentiments), 1):
    print(f"  [{score:.2f}] {text}")
print(f"  Average: {np.mean(rlhf_sentiments):.3f}\n")

print("DPO Model:")
for i, (text, score) in enumerate(zip(dpo_samples, dpo_sentiments), 1):
    print(f"  [{score:.2f}] {text}")
print(f"  Average: {np.mean(dpo_sentiments):.3f}\n")

print("=" * 80)
print(f"\nSentiment improvements:")
print(f"  RLHF: +{(np.mean(rlhf_sentiments) - np.mean(base_sentiments)):.3f}")
print(f"  DPO:  +{(np.mean(dpo_sentiments) - np.mean(base_sentiments)):.3f}")

### Visualize Method Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Base\nModel', 'RLHF\nModel', 'DPO\nModel']
avg_sentiments = [
    np.mean(base_sentiments),
    np.mean(rlhf_sentiments),
    np.mean(dpo_sentiments)
]
colors = ['gray', 'blue', 'green']

bars = ax.bar(methods, avg_sentiments, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels
for bar, value in zip(bars, avg_sentiments):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{value:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Average Positive Sentiment Score', fontsize=12)
ax.set_title('Sentiment Alignment Comparison', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1])
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Neutral')
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

## Practical Challenges in RLHF

RLHF sounds great in theory but faces several challenges in practice:

### 1. Reward Hacking / Over-Optimization

**Problem:** Model finds ways to exploit reward model weaknesses.

**Example:** If reward model favors length, model generates very long but meaningless responses.

**Solutions:**
- KL penalty (keep policy close to reference)
- Early stopping (don't over-optimize)
- Ensemble reward models
- Adversarial red-teaming

### 2. Distribution Shift

**Problem:** As policy changes, it generates text outside the reward model's training distribution.

**Solution:**
- Iterative RLHF (retrain reward model on new policy's outputs)
- Conservative KL penalties
- Robust reward model training

### 3. Sample Efficiency

**Problem:** Generating and scoring completions is expensive (forward passes through large models).

**Solutions:**
- Smaller reward models
- Cached activations
- DPO (no sampling needed during training)

### 4. Evaluation

**Problem:** How do we measure alignment? Reward model score ≠ true human preference.

**Solutions:**
- Hold-out human evaluation
- A/B testing with real users
- Multiple evaluation metrics (helpfulness, harmlessness, honesty)
- Red team testing for safety

### Demonstrating Reward Hacking

Let's show what happens without KL penalty (reward hacking).

In [ ]:
# Create model without KL penalty (β=0)
no_kl_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
no_kl_ref = GPT2LMHeadModel.from_pretrained(model_name).to(device)
no_kl_value = ValueModel(GPT2LMHeadModel.from_pretrained(model_name)).to(device)

for param in no_kl_ref.parameters():
    param.requires_grad = False
no_kl_ref.eval()

no_kl_policy_opt = torch.optim.AdamW(no_kl_model.parameters(), lr=1e-5)
no_kl_value_opt = torch.optim.AdamW(no_kl_value.parameters(), lr=1e-4)

print("Training model WITHOUT KL penalty (β=0)...\n")

# Train for a few steps
for i in range(10):
    batch_prompts = np.random.choice(ppo_prompts, size=4, replace=True).tolist()
    
    metrics = ppo_train_step(
        policy_model=no_kl_model,
        ref_model=no_kl_ref,
        value_model=no_kl_value,
        reward_model=reward_model,
        prompts=batch_prompts,
        tokenizer=tokenizer,
        policy_optimizer=no_kl_policy_opt,
        value_optimizer=no_kl_value_opt,
        beta_kl=0.0  # No KL penalty!
    )

print("\nComparing with vs without KL penalty:\n")

test_prompt = "This movie was"

print("With KL penalty (β=0.05):")
normal_samples = generate_text(policy_model, tokenizer, test_prompt, max_length=40, num_return=3)
for text in normal_samples:
    print(f"  {text}")

print("\nWithout KL penalty (β=0):")
no_kl_samples = generate_text(no_kl_model, tokenizer, test_prompt, max_length=40, num_return=3)
for text in no_kl_samples:
    print(f"  {text}")

print("\n→ Without KL penalty, model may generate repetitive or degenerate text")
print("   (though our simple example may not show dramatic differences)")

## Constitutional AI: Self-Improvement

**Constitutional AI** (CAI) is Anthropic's approach to reducing human supervision in RLHF.

### The Problem

- Collecting human preferences is expensive
- Humans may have inconsistent preferences
- Hard to scale to many different values/principles

### Constitutional AI Solution

**Phase 1: Supervised Learning (Critique + Revision)**
1. Generate response from model
2. Ask model to critique its own response based on principles
3. Ask model to revise response to address critique
4. Fine-tune on revised responses

**Phase 2: RL from AI Feedback (RLAIF)**
1. Generate pairs of responses
2. Ask model to evaluate which is better (based on constitution)
3. Train reward model on AI preferences (not human)
4. Use RL as normal

### The Constitution

A set of principles like:
- "Choose the response that is most helpful and harmless"
- "Avoid responses that are toxic, racist, or sexist"
- "Prefer responses that acknowledge uncertainty"

### Benefits

1. Less human labor needed
2. More consistent preferences (model is self-consistent)
3. Easy to update principles (just change constitution)
4. Scales better to many tasks

### Simulating Constitutional AI

We'll demonstrate the critique-revision loop (simplified).

In [ ]:
def constitutional_ai_example():
    """Demonstrate critique and revision loop."""
    
    # Constitution principle
    principle = "Generate movie reviews that are positive and enthusiastic."
    
    # Initial generation
    prompt = "This movie was"
    initial_response = generate_text(base_model, tokenizer, prompt, max_length=25, num_return=1)[0]
    
    # Critique prompt (in practice, this would be generated by the model)
    critique_prompt = f"""Response: {initial_response}
Principle: {principle}
Critique: This response"""
    
    critique = generate_text(base_model, tokenizer, critique_prompt, max_length=30, num_return=1)[0]
    
    # Revision prompt
    revision_prompt = f"""Original: {initial_response}
Principle: {principle}
Critique: {critique}
Revised:"""
    
    revised = generate_text(base_model, tokenizer, revision_prompt, max_length=25, num_return=1)[0]
    
    return {
        'principle': principle,
        'initial': initial_response,
        'critique': critique,
        'revised': revised
    }

print("Constitutional AI Demonstration\n")
print("=" * 80)

# Note: GPT-2 is not instruction-tuned, so this is illustrative only
print("\nNote: This is a simplified illustration. In practice, you'd use an")
print("instruction-tuned model that can follow critique/revision instructions.\n")

result = constitutional_ai_example()

print(f"Principle: {result['principle']}\n")
print(f"Initial response:\n  {result['initial']}\n")
print(f"Critique:\n  {result['critique']}\n")
print(f"Revised response:\n  {result['revised']}\n")

print("=" * 80)
print("\nKey idea: Model critiques and improves its own responses based on principles")
print("This reduces the need for human preference data")

## Key Takeaways

### The RLHF Pipeline

1. **SFT (Supervised Fine-Tuning)**
   - Fine-tune base LLM on demonstration data
   - Creates initial helpful assistant
   - Narrows distribution to desired behavior

2. **Reward Modeling**
   - Train model to predict human preferences
   - Uses Bradley-Terry model on pairwise comparisons
   - Learns "what makes a good response"

3. **RL with PPO**
   - Optimize policy to maximize reward
   - KL penalty prevents reward hacking
   - Balances helpfulness with coherence

### Core Concepts

- **Alignment problem**: Pre-training ≠ helpful behavior
- **Preference learning**: Humans compare, don't score
- **KL penalty**: Critical for preventing reward hacking
- **DPO**: Simpler alternative that skips reward model
- **Constitutional AI**: Self-critique reduces human labor

### RLHF vs DPO Comparison

| Aspect | RLHF (PPO) | DPO |
|--------|------------|-----|
| **Stages** | 3 (SFT → RM → RL) | 2 (SFT → DPO) |
| **Reward model** | Explicit (trained separately) | Implicit (in loss function) |
| **Optimization** | RL (PPO) | Supervised learning |
| **Stability** | Can be unstable (reward hacking) | More stable |
| **Flexibility** | Can change reward function | Fixed to training preferences |
| **Complexity** | More complex | Simpler |
| **Sample efficiency** | Less efficient (needs sampling) | More efficient |

### When to Use Each

**Use RLHF when:**
- You have a clear reward signal
- You want to iterate on reward function
- You need fine-grained control

**Use DPO when:**
- You have good preference data
- You want simpler training
- Stability is critical

## Real-World Applications

### Language Models

- **ChatGPT**: RLHF on helpfulness, harmlessness, honesty
- **Claude**: Constitutional AI for value alignment
- **Llama 2**: RLHF for safety and helpfulness

### Beyond Language

- **Code generation**: Copilot uses human preferences for code quality
- **Summarization**: Learning from human judgments of summary quality
- **Image generation**: DALL-E 3 uses human feedback for prompt following
- **Robotics**: Learning manipulation from human demonstrations

### Production Considerations

1. **Data collection**: 10k-100k preference pairs needed
2. **Annotation quality**: Inter-annotator agreement matters
3. **Infrastructure**: Distributed training, efficient inference
4. **Evaluation**: Continuous A/B testing with real users
5. **Safety**: Red team testing, adversarial evaluation

## Extensions and Future Directions

### Advanced Techniques

1. **RAFT (Reward rAnked FineTuning)**
   - Collect samples, rank by reward, fine-tune on top-k
   - Simpler than full RL

2. **Best-of-N sampling**
   - Generate N responses, return highest reward
   - No training needed, but expensive at inference

3. **Iterative RLHF**
   - Retrain reward model on new policy outputs
   - Reduces distribution shift

4. **Multi-objective RLHF**
   - Multiple reward models (helpfulness, safety, etc.)
   - Balance different objectives

5. **Process supervision**
   - Reward intermediate reasoning steps
   - Not just final answer

### Open Research Questions

- How to handle conflicting human preferences?
- Can we learn from implicit feedback (clicks, time spent)?
- How to make RLHF more sample efficient?
- How to ensure alignment generalizes to new domains?
- Can we align models to values without extensive human feedback?

## Summary

You've learned the complete RLHF pipeline that powers modern AI assistants:

1. **The alignment problem** - Pre-training alone doesn't create helpful assistants
2. **Three-stage RLHF** - SFT → Reward Modeling → RL optimization
3. **Reward modeling** - Learning to predict human preferences from comparisons
4. **PPO with KL penalty** - Optimizing for reward while staying coherent
5. **DPO alternative** - Direct preference optimization without reward model
6. **Constitutional AI** - Self-critique and improvement
7. **Practical challenges** - Reward hacking, distribution shift, evaluation

**The big picture:**

RLHF bridges the gap between language modeling and human values. It's what transformed GPT-3 (powerful but unpredictable) into ChatGPT (helpful and reliable).

The key insight: **Learning from preferences** (comparisons) is more natural and reliable than learning from absolute scores. Humans are better at saying "A is better than B" than "A deserves a score of 7.3".

This technique represents a fundamental shift: from training models to predict data to training models to satisfy human preferences and values.